# Azure ML & AI Foundry — Assignment

**Deliverables:** GitHub repo + 2-page PDF report  
**Audience:** Fresh-graduate AI engineers (independent work)

Pick **ONE** of the two tracks below and complete it end-to-end:

- **Track A** — Production-grade Azure ML pipeline (classical ML)
- **Track B** — Evaluated GenAI application with AI Foundry

Most code cells are intentionally left blank with `# TODO` comments — you fill them in.  
Library imports and Azure connections are pre-filled to save you time.

### Grading rubric (100 points)

| Points | Category | What we look for |
|---|---|---|
| 30 | Correctness | Does it run end-to-end? |
| 25 | Engineering quality | Clean code, version control, error handling |
| 25 | Evaluation rigor | Meaningful metrics, honest analysis |
| 20 | Report | Clarity, insight, what you'd do next |

# Track A — Production-grade ML Pipeline

**Goal:** Train two model versions, deploy them with blue/green traffic split, and decide which one wins on **cost vs latency vs accuracy**.

Skip this track if you picked Track B. Jump to the Track B header below.

## A.0 Setup (pre-filled — just run it)

In [ ]:
!pip install -q azure-ai-ml==1.23.0 azure-identity==1.19.0 mlflow==2.18.0 scikit-learn==1.5.2 pandas==2.2.3

In [ ]:
from azure.ai.ml import MLClient, command, Input, Output, dsl
from azure.ai.ml.entities import (
    AmlCompute,
    Data,
    Model,
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    CodeConfiguration,
)
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
import pandas as pd
import numpy as np
import os
import json
import time
import uuid

# Fill these from Azure Portal -> your Azure ML workspace -> Overview
SUBSCRIPTION_ID = "<your-subscription-id>"
RESOURCE_GROUP  = "<your-resource-group>"
WORKSPACE_NAME  = "<your-workspace-name>"

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
except Exception:
    credential = InteractiveBrowserCredential()

ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)

print("Connected to workspace:", ml_client.workspace_name)

## A.1 Pick and register your dataset

**Requirements:**
- A public tabular dataset (UCI, Kaggle, OpenML)
- Less than 100 MB
- A clear target column (regression OR classification)

**Suggestions:** Adult Income (UCI), Diabetes (sklearn), Bike Sharing (UCI), Heart Disease (UCI).

In [ ]:
# TODO: Load your dataset into a pandas DataFrame and save it to CSV.
# TODO: Register it as a versioned Data Asset.
# TODO: Print the dataset shape, the target column, and the registered asset name + version.

# Your code here



## A.2 Write component scripts

You need **at least 4 reusable components**. Suggested set:

1. `prep_data` — clean, split, encode
2. `train_model` — fit and save the model
3. `evaluate` — compute test-set metrics
4. `compare_models` — pick the winner between two trained models

Write each script to disk under `./components/<step_name>/<step>.py`.  
Use the `%%writefile` magic to create each file from a single cell.

In [ ]:
# TODO: Create component directories
# os.makedirs("components/prep", exist_ok=True)
# ... etc

# Your code here


In [ ]:
# TODO: Write prep_data.py using %%writefile components/prep/prep_data.py
# Requirements:
#   - argparse args matching the inputs/outputs you'll declare in A.3
#   - log at least one metric or print summary
#   - exit cleanly

# Your code here


In [ ]:
# TODO: Write train_model.py

# Your code here


In [ ]:
# TODO: Write evaluate.py

# Your code here


In [ ]:
# TODO: Write compare_models.py
# This component takes two model outputs + their metrics, and writes a decision JSON file
# saying which one to promote and why.

# Your code here


## A.3 Define the components in SDK v2

Use `azure.ai.ml.command(...)` for each script you wrote in A.2.

In [ ]:
# This environment works for all four components — leave it as-is
ENV = "azureml://registries/azureml/environments/sklearn-1.5/labels/latest"

# TODO: Define four command components.
# prep_component    = command(...)
# train_component   = command(...)
# eval_component    = command(...)
# compare_component = command(...)

# Your code here


## A.4 Assemble pipeline v1 (full data)

Train one version of the model on the **full** dataset.

In [ ]:
# TODO: Build a @dsl.pipeline that chains prep -> train -> evaluate on the full data.
# TODO: Submit the pipeline job.
# TODO: Register the trained model as <your-model-name> version 1.

# Your code here


## A.5 Assemble pipeline v2 (subset)

Train a second version on a **30% subset** of the training data — same algorithm, fewer rows. This simulates an experimental challenger model.

In [ ]:
# TODO: Modify prep_data to accept a sample_fraction parameter,
# OR take a random 30% sample of the registered data asset before training.
# TODO: Submit the v2 pipeline and register the model as version 2.

# Your code here


## A.6 Deploy both versions with blue/green traffic split

Requirements:

- `blue` deployment serves model **v1** with 90% traffic
- `green` deployment serves model **v2** with 10% traffic
- Both deployments live under the **same endpoint**
- Configure autoscaling (min 1, max 3 instances, CPU > 70% scale-up)

In [ ]:
ENDPOINT_NAME = f"track-a-{uuid.uuid4().hex[:6]}"

# TODO: Create one managed online endpoint named ENDPOINT_NAME.
# TODO: Write a single score.py (under ./scoring/) that both deployments will use.
# TODO: Create deployment "blue" pointing at model v1.
# TODO: Create deployment "green" pointing at model v2.
# TODO: Set traffic split to {"blue": 90, "green": 10}.
# TODO: Configure autoscaling on the blue deployment.

# Your code here


## A.7 Capture P50, P95, P99 latency

Send at least 200 requests to the endpoint (mix random rows from your test set). Record per-call latency and plot a histogram.

In [ ]:
# TODO: Send >= 200 calls to the endpoint, capture per-call latency in ms.
# TODO: Compute P50, P95, P99.
# TODO: Plot a histogram with matplotlib.
# Hint: use ml_client.online_endpoints.invoke() in a loop.

# Your code here


## A.8 Compare quality, latency, and cost

Build a one-row-per-model comparison table with these columns:

| Model | Test RMSE / Accuracy | Latency P95 | Estimated cost / 1k calls |

For cost, estimate as `(instance_count × VM hourly rate × time hours)` + endpoint base cost. Look up the price of `Standard_DS3_v2` in your region on the Azure pricing page.

In [ ]:
# TODO: Combine evaluation metrics from A.4 and A.5 with latency from A.7.
# TODO: Build a pandas DataFrame and display it.

# Your code here


## A.9 Pick a winner — and justify it

Answer in 3–4 sentences in the markdown cell below.

- Which deployment would you promote to 100%?
- Did the smaller v2 model trade off enough quality for the cost or latency win?
- What additional signal would make this decision easier?

**Your answer:**

_Write your decision here._

## A.10 Cleanup

In [ ]:
# TODO: Delete the endpoint to stop billing.
# ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()

# Your code here


# Track B — Evaluated GenAI Application

**Goal:** Build a RAG app over your own knowledge base, then evaluate and red-team it like a real production system.

Skip this track if you picked Track A.

## B.0 Setup (pre-filled — just run it)

In [ ]:
!pip install -q azure-ai-projects==1.0.0 azure-ai-evaluation==1.5.0 azure-ai-inference==1.0.0b9 openai==1.55.0

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.evaluation import (
    evaluate,
    GroundednessEvaluator,
    RelevanceEvaluator,
    CoherenceEvaluator,
    FluencyEvaluator,
    SimilarityEvaluator,
    HateUnfairnessEvaluator,
    ViolenceEvaluator,
)
from azure.identity import DefaultAzureCredential
import json
import os
import time

# Fill from Foundry portal -> your project -> Overview
PROJECT_ENDPOINT = "https://<your-foundry-account>.services.ai.azure.com/api/projects/<your-project>"

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

# Judge LLM used by the quality evaluators
JUDGE = {
    "azure_endpoint":   "https://<your-foundry-account>.openai.azure.com/",
    "api_key":          "<your-key>",
    "azure_deployment": "gpt-4o",
    "api_version":      "2024-10-21",
}

print("Connected to Foundry project.")

## B.1 Pick a domain and gather 10–20 documents

Pick a **domain** that interests you — legal FAQ, medical first-aid, customer support, education, internal HR — your choice.

Gather **10–20 short documents** (plain text or PDF excerpts of under 500 words each). Put them in a Python dictionary `DOCS = {"doc_id": "text", ...}` or load from files.

In [ ]:
# TODO: Define your DOCS dictionary or load files into one.
# TODO: Write a retriever function retrieve(query, k=3) that returns the top-k docs.
# Hint: keyword overlap is fine; you may also use sentence-transformers.

DOCS = {
    # "doc1": "...",
    # "doc2": "...",
}

def retrieve(query, k=3):
    # Your code here
    pass


## B.2 Build the RAG flow

The `ask(query, model_name)` function should:

1. Retrieve top-k docs with your retriever
2. Build a prompt with the context
3. Call the LLM
4. Return a dict with keys: `query`, `response`, `context`, `ground_truth` (leave ground_truth blank for now)

In [ ]:
def ask(query, model_name="gpt-4o-mini"):
    # TODO: Get the OpenAI client:
    #   client = project.inference.get_azure_openai_client(api_version="2024-10-21")
    # TODO: Retrieve context, build messages, call client.chat.completions.create(...)
    # TODO: Return the structured dict described above
    pass

# Smoke test (uncomment when ready)
# print(ask("your test question here"))


## B.3 Hand-author a 20-row evaluation dataset

Your dataset must contain a mix of:

- **10 happy-path** questions — answers should be in your docs
- **5 edge cases** — empty input, very long input, ambiguous phrasing, multi-language
- **5 adversarial** — prompt injection attempts, off-topic asks, role-play tricks

Each row must have `query`, `response`, `context`, `ground_truth`.

In [ ]:
# TODO: Build a list of 20 (query, ground_truth) tuples.
# TODO: For each query, call ask() to populate response + context.
# TODO: Write the result to eval_dataset.jsonl.

QUERIES = [
    # ("question 1", "expected substring or sentence"),
    # ...
]

# Your code here


## B.4 Run all 5 quality + 2 safety evaluators

Required evaluators:

- **Quality (5):** Groundedness, Relevance, Coherence, Fluency, Similarity
- **Safety (2):** HateUnfairness, Violence

In [ ]:
# TODO: Call evaluate(...) with all 7 evaluators on eval_dataset.jsonl.
# TODO: Save the per-row results to eval_results.json.
# TODO: Print the aggregate metrics dictionary.

# Your code here


## B.5 Write ONE custom evaluator

Pick one of:

- **Response length** — flag responses outside 20–300 words
- **Tone match** — match a target tone (formal / friendly) using the judge LLM
- **Language match** — verify the response is in the same language as the question

A custom evaluator is just a callable that takes `**kwargs` and returns a dict of scores.

In [ ]:
class MyCustomEvaluator:
    """TODO: implement your custom evaluator."""

    def __init__(self, **kwargs):
        # Initialize anything you need (LLM client, thresholds, ...)
        pass

    def __call__(self, *, query, response, **kwargs):
        # Return a dict like {"my_score": 0.85, "my_score_reason": "..."}
        # Your code here
        return {}

# TODO: Re-run evaluate() including MyCustomEvaluator and inspect the new column.


## B.6 Red Teaming Agent — find 3 vulnerabilities

Use the Foundry AI Red Teaming Agent to probe your app. Document 3 attacks that succeeded (fully or partially).

In [ ]:
# TODO: Use azure.ai.evaluation.red_team.RedTeam to run a scan.
# TODO: Configure target = your ask() function.
# TODO: Capture the report and save it to red_team_report.json.

# Your code here


**Vulnerabilities you found:**

1. _Attack type and what happened…_
2. _Attack type and what happened…_
3. _Attack type and what happened…_

**For each vulnerability, what would you change in the prompt or system to fix it?**

_Your answers here._

## B.7 Compare two models

Re-run the evaluation with `gpt-4o` as the target model (instead of `gpt-4o-mini`). Build a side-by-side comparison table:

| Model | Avg Groundedness | Avg Relevance | Latency P95 | Tokens per call |

In [ ]:
# TODO: Run the same eval set against both gpt-4o-mini and gpt-4o.
# TODO: Record latency and token counts during inference.
# TODO: Build the comparison DataFrame.

# Your code here


## B.8 Reflection — where does your app shine and break?

Answer in 4–6 sentences below:

- What kinds of queries does it handle well?
- What kinds break it?
- Which evaluator caught the most real problems?
- If you had another week, what would you fix first?

**Your reflection:**

_Write your reflection here._

# Deliverables Checklist

Before you submit, verify:

- [ ] GitHub repo is public or shared with the instructor
- [ ] README explains how to set up and run your notebook
- [ ] Screenshots from the Foundry portal or Azure ML Studio are in `/screenshots`
- [ ] Your 2-page reflection PDF is in the repo root as `REPORT.pdf`
- [ ] All Azure resources you created have been **cleaned up** (no orphan endpoints!)

**Submission deadline:** one week from today.

Good luck — build something you would be proud to show in an interview.